In [ ]:
# Install dependencies
!pip install -q torch torchvision scikit-learn scikit-image opencv-python tqdm pandas pillow

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

print('\n✓ Setup complete!')

In [ ]:
# Step 2: Verify Data (data already in Kaggle dataset)
import os
import shutil

print("📋 Checking for data...\n")

# Path to your uploaded dataset
dataset_path = '/kaggle/input/datasets/akshitarora345/cervical-cancer-dataset/data'
local_data_path = 'data'

if os.path.exists(dataset_path):
    print(f"✅ Found dataset at: {dataset_path}\n")
    
    # Copy to local working directory for easier access
    if not os.path.exists(local_data_path):
        print(f"📦 Copying data to working directory...")
        shutil.copytree(dataset_path, local_data_path)
        print("✅ Data copied!")
    
    print("\n📊 Data structure:")
    for root, dirs, files in os.walk(local_data_path):
        level = root.replace(local_data_path, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 2 * (level + 1)
        file_count = len([f for f in files if f.endswith(('.png', '.jpg', '.jpeg'))])
        if file_count > 0:
            print(f'{subindent}{file_count} images')
    
    print("\n✓ Data ready for synthetic generation!")
else:
    print(f"❌ Dataset not found at {dataset_path}")
    print("\nTrying alternative paths...")
    
    # Try to find data folder
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'data' in dirs:
            print(f"Found data at: {os.path.join(root, 'data')}")

In [ ]:
# Verify data counts
import os

def count_images(directory):
    classes = {}
    for class_name in os.listdir(directory):
        class_path = os.path.join(directory, class_name)
        if os.path.isdir(class_path):
            count = len([f for f in os.listdir(class_path) if f.endswith(('.png', '.jpg'))])
            classes[class_name] = count
    return classes

print('Training Data:')
train_counts = count_images('data/train')
for cls, count in sorted(train_counts.items()):
    print(f'  {cls}: {count}')

print('\nValidation Data:')
val_counts = count_images('data/val')
for cls, count in sorted(val_counts.items()):
    print(f'  {cls}: {count}')

print(f'\nTotal Training: {sum(train_counts.values())}')
print(f'Total Validation: {sum(val_counts.values())}')
print('\n✓ Data verified!')

In [ ]:
# GPU-OPTIMIZED TRAINING with Automatic Mixed Precision (AMP)
print('🚀 GPU Training: Mixed Precision + Larger Batch Size')
print('Target: 75-85% Accuracy with balanced class detection')
print('Time: ~30-45 minutes (vs 2+ hours on CPU)')
print('=' * 80)

!cd /content && python backend/train_hybrid.py \
    --data-dir data \
    --epochs 100 \
    --batch-size 64 \
    --learning-rate 0.0003 \
    --early-stopping-patience 25 \
    --num-workers 4 \
    --checkpoint-dir checkpoints

In [ ]:
# Check training results
import os

print('Training Complete!\n')
print('Generated checkpoints:')
!ls -lh checkpoints/*.pth

# Show best model info
if os.path.exists('checkpoints/best_hybrid_model.pth'):
    import torch
    checkpoint = torch.load('checkpoints/best_hybrid_model.pth', map_location='cpu', weights_only=False)
    print(f"\n✓ Best Model:")
    print(f"  Validation Accuracy: {checkpoint.get('val_accuracy', 'N/A'):.2f}%")
    print(f"  Epoch: {checkpoint.get('epoch', 'N/A')}")
    print(f"  Classes: {checkpoint.get('num_classes', 'N/A')}")
else:
    print('\n⚠ Best model not found!')

In [ ]:
# Download trained model
from google.colab import files
import os

# Zip all checkpoints
!zip -r trained_model.zip checkpoints/

print('Downloading trained model...')
files.download('trained_model.zip')

print('\n✓ Download started!')
print('Extract on your Mac and copy to:')
print('/Users/akshitarora/cervical-cancer-classifier-local/backend/checkpoints/')

---
## Next Steps on Your Mac:
